<img src='https://gitlab.eumetsat.int/eumetlab/oceans/ocean-training/tools/frameworks/-/raw/main/img/MSOC_banner.png' align='right' width='100%'/>

<a href="../Index.ipynb">Index</a><br>

**Copyright:** 2026 European Union <br>
**License:** MIT <br>
**Authors:** Ben Loveday (EUMETSAT/Innoflair UG), Hayley Evers-King (EUMETSAT), Juan Ignacio-Gossn (EUMETSAT)

<div class="alert alert-block alert-success">
<h3>Multi-sensor Ocean Colour Course</h3></div>

<div class="alert alert-block alert-warning">
    
<b>PREREQUISITES </b>
    
This notebook has the following prerequisites:
- **<a href="https://user.eumetsat.int/register" target="_blank">A EUMETSAT User Portal account</a>** if you want to download Sentinel-3 OLCI marine data from the EUMETSAT Data Store

There are no prerequisite notebooks for this module.
</div>
<hr>

# Single sensor validation of OLCI with the ThoMaS match-up toolkit

### Data used

| Dataset | Source / ID | product description |
|:--------------------:|:-----------------------:|:-------------:|
| Sentinel-3 OLCI level 2 full resolution (operational) | EUMETSAT / EO:EUM:DAT:0407 | <a href="https://user.eumetsat.int/catalogue/EO:EUM:DAT:SENTINEL-3:OL_2_WFR___NTC" target="_blank">Description</a> |
| Sentinel-3 OLCI level 2 full resolution (version BC003) reprocessing | EUMETSAT / EO:EUM:DAT:0556 | <a href="https://user.eumetsat.int/catalogue/EO:EUM:DAT:0556" target="_blank">Description</a> | EO:EUM:DAT:SENTINEL-3:0556 |


### Learning outcomes

At the end of this notebook you will know;
* how to use the ThoMaS toolkit to perform single sensor match-up validation extractions and analyses

### Outline

Validation studies are essential to ensuring satellite sensor performance and a necessary part of algorithm development. For ocean colour studies, match-up analyses are a key part of validation. The ThoMaS (Tool to generate Matchups of OC products with Sentinel-3/OLCI) package provides a comprehensive set of tools to help with the validation of ocean colour products for various sensors, including Sentinel-3 OLCI, MODIS Aqua and PACE OCI, supporting many common workfows including;
* satellite data acquisition
* mini file extraction
* *in situ* data management
* BRDF correction

ThoMaS is written in Python and is made available through a **<a href="https://gitlab.eumetsat.int/eumetlab/oceans/ocean-science-studies/ThoMaS" target="_blank">EUMETSAT Gitlab repository</a>**. The package can be used from the command line, via GUI, or imported as a Python library, as done here. This notebook contains a single example of how to use ThoMaS, but the capability shown is not exhaustive. Many more command-line examples are included in the repository, and we encourage users to familiarise themselves with both the **<a href="https://gitlab.eumetsat.int/eumetlab/oceans/ocean-science-studies/ThoMaS/-/blob/main/README.md" target="_blank">project README</a>** and **<a href="https://gitlab.eumetsat.int/eumetlab/oceans/ocean-science-studies/ThoMaS/-/blob/main/README_examples.md" target="_blank"> example README</a>** for more information.

<div class="alert alert-info" role="alert">

## Importing dependencies

</div>

We begin by importing all of the libraries that we need to run this notebook. If you have built your python environment (`msoc`) using the file provided in this repository, then you should have everything you need. For more information on building environment, please see the repository **<a href="../README.md" target="_blank">README</a>**.

In [1]:
import os                                   # a library that allows us access to basic operating system commands like making directories
import sys                                  # a library that provides access to system commands
import pandas as pd                         # a library that helps us manipulate data 
import shutil                               # a library that allows us access to basic operating system commands like copy
import numpy as np                          # a library that provides support for array-based mathematics
import glob                                 # a library that helps us find files
import image_carousel                       # a bespoke function to make an image carousel
from pathlib import Path                    # a library to help us manage paths

Everything is now imported, we can proceed...

<div class="alert alert-warning" role="alert">

## Defining functions

</div>

Before we go any further we are going to define two quick functions that helps u to write our configuration options to a file and find the git root of this repository.

In [2]:
# Write config_params sections into config_file.ini
def write_config_file(path_to_config_file,config_params):
    with open(path_to_config_file, 'w') as text_file:
        for section,section_params in config_params.items():
            text_file.write('\n[%s]\n' % (section))
            for param, value in section_params.items():
                text_file.write('%s: %s\n' % (param, value))

In [3]:
def find_git_root(start_path=None):
    path = Path(start_path or os.getcwd()).resolve()
    for parent in [path] + list(path.parents):
        if (parent / ".git").exists():
            return parent
    return None

<div class="alert alert-info" role="alert">

## Configuring the ThoMaS toolkit

</div>

The **ThoMaS** toolkit is included as a submodule in this repository. To import it, we just need to make sure that Python can find the correct path to it. Lets find and import ThoMaS below.

In [4]:
thomas_path = os.path.join(find_git_root(), "3_courses", "msoc", "submodules", "ThoMaS")
sys.path.append(thomas_path)
from main import ThoMaS_main as ThoMaS

If, for any reason, you cannot find ThoMaS in the submodule folder of this repository, you can download thomas using:

`git clone --depth 1 https://gitlab.eumetsat.int/eumetlab/oceans/ocean-science-studies/ThoMaS.git`

You will need to use the cell above to include ThoMas in the system path (using `sys.path.append`).

Now we should have ThoMaS imported and configured for use, so we can proceed with our examples.

<div class="alert alert-info" role="alert">

## Example: Full match-up exercise using MOBY SeaBASS data (adapted from <a href="https://gitlab.eumetsat.int/eumetlab/oceans/ocean-science-studies/ThoMaS/-/blob/main/README_examples.md?ref_type=heads" target="_blank">ThoMaS example 6</a>)
[Back to top](#Contents)

</div>

<div class="alert alert-success" role="alert">

Run ThoMaS for:

1. full matchup exercise: satellite extractions + minifiles + extraction statistics + matchup statistics
1. insitu data already available in SeaBASS/OCDB format

</div>

#### Description
<hr>

In this case we are going to look at a full match-up analysis using a hyperspectral *in situ* measurements from Moby, in SEABASS format, and stored in a local database. The measurements are not corrected for BRDF effects. 

1. I have prepared a set of hyperspectral Rrs insitu measurements from <a href="https://mlml.sjsu.edu/moby/" target="_blank">MOBY</a> in SeaBASS format not corrected for BRDF effects.
1. I wish to get matchups between this MOBY subset and S3A/OLCI standard FR Level 2 SatData:
    * using some kind of extraction protocol....,
    * an extraction window of 5x5,
    * an insitu-satellite time difference threshold of 1 hour (7200 seconds).
    * to propagate - if existent - "uncertainties" in insitu and/or satellite data to the performance metrics by running Monte-Carlo simulations. NB: For the moment, the "uncertainty" in the satellite data is usually taken as the standard deviation of the set of valid macropixels.
1. I am not interested in getting ancillary data from ECMWF corresponding to the insitu data.
1. I want to apply the O25 BRDF correction to the insitu data.
1. I may have several insitu measurements corresponding to one single SatData within the time window that I selected, but I wish to keep only the closest in time with the satellite overpass.
1. I wish my satellite data to be stored in the ./MOBY/SatData directory. All the other outputs (IDB, minifiles, EDB, MDB, etc.) to be stored in the ./MOBY directory

<hr>

First off, lets create a directory to store our experiment in.

In [5]:
# Define and create the path to your main output directory for this exercise.
example_tag = "MOBY"
output_path = os.path.join(os.getcwd(), example_tag)
if not os.path.exists(output_path):
    os.mkdir(output_path)

Lets now copy our example SEABASS/OCDB format *in situ* data into this experiment path.

In [6]:
# Copy the MOBY sample data (/ThoMaS/examples/examples6/MOBY_OCDB.csv) to your target output directory
path_insitu = os.path.join(output_path, 'MOBY_OCDB.sb')
shutil.copy(os.path.join(thomas_path, "examples", "06_full_matchup_workflow_MOBY_SeaBASS", "MOBY", "MOBY_OCDB.sb"), path_insitu);

Lets take a quick look at a subset of the data....

In [7]:
with open(path_insitu) as file:
    print('\t'.join(["Station", "\t\tDate", "\tTime", "\tStatus"]))
    for line in file:
        if not any(ext in line[0] for ext in ['/', '!']):
            print('\t'.join(line.rstrip().split(',')[0:4]))

Station			Date		Time		Status
M260_2016061020d	20160610	21:14:37	Good
M260_2016071120d	20160711	21:14:32	Good
M260_2016071920d	20160719	21:15:05	Questionable
M260_2016072220d	20160722	21:14:37	Questionable
M260_2016080320d	20160803	21:14:32	Good
M261_2016082620d	20160826	21:13:24	Good
M261_2016093020d	20160930	21:13:54	Good
M261_2016100820d	20161008	21:13:27	Good
M261_2016101920d	20161019	21:15:10	Good
M261_2017010420d	20170104	21:06:21	Good
M261_2017010520d	20170105	21:06:28	Good
M261_2017011120d	20170111	21:05:57	Good
M261_2017011220d	20170112	21:06:00	Good
M262_2017022320d	20170223	21:15:58	Questionable
M262_2017031520d	20170315	21:15:54	Questionable
M262_2017032620d	20170326	21:15:56	Good
M262_2017040320d	20170403	21:14:46	Good
M262_2017052320d	20170523	21:17:10	Good
M262_2017052720d	20170527	21:16:57	Good
M262_2017060420d	20170604	21:17:47	Good
M262_2017061920d	20170619	21:16:51	Questionable
M263_2017080420d	20170804	21:14:21	Good
M263_2017082020d	20170820	21:12:26	Questionable
M26

We will define everything we need to do in a ThoMaS configuration file. However, there are some differences to before. We need to define all the steps in our analysis, including:

* how we handle the *in situ* data
* how we handle the satellite data
* how we handle the generation of satellite minifiles
* how we handle the construction of the extraction data base
* how we handle the generation of the match-up database (MDB)

Lets set up our configuration file to specify all the relevant steps in our desired workflow.

In [8]:
# Build your config_file.ini
path_to_config_file = os.path.join(output_path, 'config_file.ini')
config_params = {}

# global
config_params['global'] = {}
config_params['global']['SetID'] = 'MOBY'
config_params['global']['path_output'] = output_path
config_params['global']['overwrite'] = 'all_except_SatData_minifiles' # Don't change this parameter unless you want to download data again!

# workflow
config_params['workflow'] = {}
config_params['workflow']['workflow'] = 'insitu, SatData, minifiles, EDB, MDB'

# insitu
config_params['insitu'] = {}
config_params['insitu']['insitu_data2OCDBfile'] = None
config_params['insitu']['insitu_input'] = path_insitu
config_params['insitu']['insitu_satellite_time_tolerance_seconds'] = 3600
config_params['insitu']['insitu_getAncillary'] = False 
config_params['insitu']['insitu_BRDF'] = 'O25'

# satellite
config_params['satellite'] = {}
config_params['satellite']['satellite_source'] = 'online'
config_params['satellite']['satellite_path-to-SatData'] = os.path.join(output_path, 'SatData')
config_params['satellite']['satellite_collections'] = 'operational'
config_params['satellite']['satellite_platforms'] = 'S3A'
config_params['satellite']['satellite_levels'] = 'L2'
config_params['satellite']['satellite_resolutions'] = 'FR'
config_params['satellite']['satellite_BRDF'] = 'O25'
config_params['satellite']['satellite_download_option'] = 'standard'

# minifiles
config_params['minifiles'] = {}
config_params['minifiles']['minifiles_winSize'] = 5

# EDB
config_params['EDB'] = {}
config_params['EDB']['EDB_protocols_L2'] = 'noOutliers_noScreening'
config_params['EDB']['EDB_winSizes'] = 5

# MDB
config_params['MDB'] = {}
config_params['MDB']['MDB_time-interpolation'] = 'insitu2satellite_NN'
config_params['MDB']['MDB_stats_MonteCarlo'] = 100
config_params['MDB']['MDB_stats_plots'] = True
config_params['MDB']['MDB_matchup_by_matchup_plots'] = False
config_params['MDB']['MDB_stats_protocol'] = 'EUMETSAT_standard_L2'

# Write config_params sections into config_file.ini
write_config_file(path_to_config_file, config_params)

Lets run this configuration (which will take some time)...

In [9]:
# Run ThoMaS and check all the outputs in the output directory
ThoMaS(path_to_config_file)


Running MOBY

Building satellite_datasets file from specified options in config_file...

Step insitu
Creating IDB (in situ database) netcdf file and calculating timeRanges and satellite datasets from input SeaBASS/OCDB file (in situ)
insitu: Spectral matching for sensor OLCI and platform S3A
File not altered by sanitiser, skipping
Building insitu netCDF file for insitu set MOBY, spectrally matched to platform S3A, sensor OLCI

Step SatData
Writing SatDataList for set S3A_LOFTY_OLCI_L2_IPF_OL__L2M.003_FR_MOBY to: /Users/benloveday/Code/git_repositories/CMTS/internal/ocean/applications/ocean-colour-applications/3_courses/msoc/Day5/MOBY/SatDataLists/SatDataList_S3A_LOFTY_OLCI_L2_IPF_OL__L2M.003_FR_MOBY.txt

Step minifiles
Extracting minifiles from satellite data
Creating minifile: S3A_OL_2_WFR____20160610T202635_20160610T202835_20210706T132838_0119_005_128______MAR_R_NT_003.SEN3.LOFTY_W15719016N2081716_5x5.nc
Creating minifile: S3A_OL_2_WFR____20160711T202303_20160711T202503_20210707T002

All done! Let's check our outputs. In this case, we have run a full analysis, so along side all of our data we have some summary plots. Lets view some...

In [10]:
image_paths = glob.glob(os.path.join(os.getcwd(), example_tag, "MDB", "plots_global", "*.png"))
carousel = image_carousel.create_image_carousel(image_paths, height=800, width=1200)
carousel

<div class="alert alert-danger" role="alert">

**WAIT! How does this analysis look?**

* Does the match-up analysis suggest a good agreement?
* Are we sure that we have set the parameters of the analysis as well as we can? 
</div>

<div class="alert alert-success" role="alert">

## What next?

Take a closer look at the paramters we chose, and use what you learned from the earlier presentations to see if you can improve the quality of the analysis. If you have validation data for your ROI, then perhaps you can adapt the above for your own area.
</div>

<hr>
<a href="../Index.ipynb">Index</a>
<hr>
<a href="https://gitlab.eumetsat.int/eumetlab/ocean">View on GitLab</a> | <a href="https://training.eumetsat.int/">EUMETSAT Training</a> | <a href=mailto:ops@eumetsat.int>Contact helpdesk for support </a> | <a href=mailto:.training@eumetsat.int>Contact our training team to collaborate on and reuse this material</a></span></p>